# Stop Locations Pre-processing
This notebook handles the cleaning, transformation, and logical deduplication of the stop locations dataset.

In [ ]:
import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv('<RAW_DATA_DIR>/stop_locations.csv')
print(f"Initial shape: {df.shape}")
df.head()

## 1. Check for missing coordinates and geocode if necessary

In [ ]:
missing_coords = df[df['lat'].isna() | df['lng'].isna()]
print(f"Total missing coordinates: {len(missing_coords)}")

if len(missing_coords) > 0:
    df['geocoded_flag'] = 0  # Initializing flag
    print("Warning: Found records with missing coordinates. Flagging for manual check.")
    df.loc[missing_coords.index, 'geocoded_flag'] = 1

## 2. Normalization & Parent Stop Mapping
Handle intersection sorting and map logical duplicates to a single `parent_stop_id`.

In [ ]:
def normalize_intersection(addr):
    if not isinstance(addr, str):
        return addr
    # Handle intersection patterns like "A & B" or "A / B"
    pattern = r'\s+(&|and|\/)\s+'
    if re.search(pattern, addr, re.IGNORECASE):
        main_part = addr.split(',')[0]
        rest = addr[len(main_part):]
        res = re.split(r'(\s+(&|and|\/)\s+)', main_part, flags=re.IGNORECASE)
        if len(res) >= 3:
            s1, sep, s2 = res[0], res[1], res[3]
            streets = sorted([s1.strip(), s2.strip()])
            return f'{streets[0]}{sep}{streets[1]}{rest}'
    return addr

# 1. Normalize Addresses
df['normalized_address'] = df['address'].apply(normalize_intersection)

# 2. Map Parent Stop ID (Grouping by coordinates and normalized address)
df['parent_stop_id'] = df.groupby(['lat', 'lng', 'normalized_address'])['stop_location_id'].transform('min')

# 3. Identify Logical Duplicates (Binary 1/0)
df['is_logical_duplicate'] = (df['stop_location_id'] != df['parent_stop_id']).astype(int)

print(f"Total logical duplicate records detected: {df['is_logical_duplicate'].sum()}")

if df['is_logical_duplicate'].any():
    print("\nList of Logical Duplicate Stops (for manual review):")
    cols = ['stop_location_id', 'parent_stop_id', 'is_logical_duplicate', 'address', 'normalized_address', 'lat', 'lng']
    display(df[df['is_logical_duplicate'] == 1][cols].sort_values(by='parent_stop_id'))

## 3. Feature Engineering
- Map `is_school` to binary (1/0) directly
- Define `geofence_radius_m` (80m or 300m)
- Create `coordinates` tuple column
- Extract City, State, Zip

In [ ]:
def parse_address_components(addr):
    if not isinstance(addr, str):
        return None, "MN", None
    parts = [p.strip() for p in addr.split(',')]
    if len(parts) >= 3:
        city = parts[-2]
        state_zip = parts[-1].split(' ')
        state = "MN"
        zip_code = None
        for part in state_zip:
            if part == "MN":
                state = "MN"
            elif part.isdigit() or '-' in part:
                zip_code = part
        return city, state, zip_code
    return None, "MN", None

# Extract components
df[['city', 'state', 'postal_code']] = df['address'].apply(lambda x: pd.Series(parse_address_components(x)))

# Convert is_school to binary
df['is_school'] = df['is_school'].apply(lambda x: 1 if str(x).lower() == 'true' else 0)

# Geofence radius
df['geofence_radius_m'] = df['is_school'].apply(lambda x: 300 if x == 1 else 80)

df.head()

## 4. Remove duplicate locations

In [ ]:
# Keep only one row per parent_stop_id
# 'keep=first' ensures we retain the original record that served as the parent
df_unique = df.drop_duplicates(subset=['parent_stop_id'], keep='first').copy()

# Print summary of the cleaning process
print(f"Original records: {len(df)}")
print(f"Logical duplicates removed: {df['is_logical_duplicate'].sum()}")
print(f"Final unique locations: {len(df_unique)}")

## 4. Export Cleaned Data

In [ ]:
output_path = '<PROCESSED_DATA_DIR>/stop_locations_cleaned.csv'
df_unique.to_csv(output_path, index=False)
print(f"Successfully exported to {output_path}")